In [1]:
import pandas as pd
from pathlib import Path

print("Pandas version:", pd.__version__)

Pandas version: 3.0.5


In [2]:
# Project directory
BASE_DIR = Path.cwd()

if not (BASE_DIR / "data").exists():
    BASE_DIR = BASE_DIR.parent

# Dataset path
DATA_FILE = BASE_DIR / "data" / "sales_data.csv"

# Load dataset
df = pd.read_csv(DATA_FILE, encoding="utf-8")

print("Dataset loaded successfully!")

Dataset loaded successfully!


In [3]:
df.head()

,OrderID,OrderDate,CustomerID,Product,Category,Quantity,UnitPrice,Revenue,Region,Status
0,1001,2026-10-17,C0063,Mouse,Accessories,1,20.97,20.97,South,Completed
1,1002,2025-04-15,C0009,Pen,Stationery,1,2.04,2.04,North,Completed
2,1003,2025-08-27,C0051,Pen,Stationery,5,1.81,9.05,West,Completed
3,1004,2026-08-27,C0041,Headphones,Electronics,1,105.18,105.18,West,Completed
4,1005,2025-06-09,C0098,Monitor,Electronics,3,184.09,552.27,North,Completed


In [4]:
df.shape

(1000, 10)

In [5]:
df.columns.tolist()

['OrderID',
 'OrderDate',
 'CustomerID',
 'Product',
 'Category',
 'Quantity',
 'UnitPrice',
 'Revenue',
 'Region',
 'Status']

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   OrderID     1000 non-null   int64  
 1   OrderDate   1000 non-null   str    
 2   CustomerID  1000 non-null   str    
 3   Product     1000 non-null   str    
 4   Category    1000 non-null   str    
 5   Quantity    1000 non-null   int64  
 6   UnitPrice   1000 non-null   float64
 7   Revenue     1000 non-null   float64
 8   Region      1000 non-null   str    
 9   Status      1000 non-null   str    
dtypes: float64(2), int64(2), str(6)
memory usage: 122.0 KB


In [ ]:
هدف ما فقط اجرای کد نیست؛ می‌خواهیم یاد بگیریم چطور یک Data Analyst کیفیت داده‌های مشتری را قبل از تحلیل و ساخت داشبورد ارزیابی می‌کند.

Phase 2 — بخش دوم: Missing Values Analysis

In [7]:
# Count missing values in each column
missing_values = df.isnull().sum()

print("Missing Values:")
print(missing_values)

Missing Values:
OrderID       0
OrderDate     0
CustomerID    0
Product       0
Category      0
Quantity      0
UnitPrice     0
Revenue       0
Region        0
Status        0
dtype: int64


In [8]:
# Calculate missing value percentage
missing_percentage = (df.isnull().sum() / len(df)) * 100

print(missing_percentage)

OrderID       0.0
OrderDate     0.0
CustomerID    0.0
Product       0.0
Category      0.0
Quantity      0.0
UnitPrice     0.0
Revenue       0.0
Region        0.0
Status        0.0
dtype: float64


In [9]:
# Count duplicate rows
duplicate_rows = df.duplicated().sum()

print("Duplicate Rows:", duplicate_rows)

Duplicate Rows: 0


In [10]:
# Check duplicate OrderID
duplicate_orders = df["OrderID"].duplicated().sum()

print("Duplicate OrderIDs:", duplicate_orders)

Duplicate OrderIDs: 0


In [11]:
duplicate_data = df[
    df["OrderID"].duplicated(keep=False)
]

duplicate_data

,OrderID,OrderDate,CustomerID,Product,Category,Quantity,UnitPrice,Revenue,Region,Status


In [12]:
print("Invalid Quantity:")
print((df["Quantity"] <= 0).sum())

print("Invalid UnitPrice:")
print((df["UnitPrice"] < 0).sum())

print("Invalid Revenue:")
print((df["Revenue"] < 0).sum())

Invalid Quantity:
0
Invalid UnitPrice:
0
Invalid Revenue:
0


In [13]:
expected_revenue = df["Quantity"] * df["UnitPrice"]

revenue_errors = (
    (df["Revenue"] - expected_revenue).abs() > 0.01
)

print("Revenue Calculation Errors:", revenue_errors.sum())

Revenue Calculation Errors: 0


مرحله بعد: Data Type Conversion

In [14]:
df["OrderDate"] = pd.to_datetime(df["OrderDate"])

print(df["OrderDate"].dtype)

datetime64[us]


In [15]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   OrderID     1000 non-null   int64         
 1   OrderDate   1000 non-null   datetime64[us]
 2   CustomerID  1000 non-null   str           
 3   Product     1000 non-null   str           
 4   Category    1000 non-null   str           
 5   Quantity    1000 non-null   int64         
 6   UnitPrice   1000 non-null   float64       
 7   Revenue     1000 non-null   float64       
 8   Region      1000 non-null   str           
 9   Status      1000 non-null   str           
dtypes: datetime64[us](1), float64(2), int64(2), str(5)
memory usage: 112.2 KB


In [16]:
df["Year"] = df["OrderDate"].dt.year

df["Month"] = df["OrderDate"].dt.month

df["YearMonth"] = df["OrderDate"].dt.to_period("M").astype(str)

df[["OrderDate", "Year", "Month", "YearMonth"]].head()

,OrderDate,Year,Month,YearMonth
0,2026-10-17,2026,10,2026-10
1,2025-04-15,2025,4,2025-04
2,2025-08-27,2025,8,2025-08
3,2026-08-27,2026,8,2026-08
4,2025-06-09,2025,6,2025-06


Phase 2 — Final Step: Data Quality Report

In [17]:
quality_report = pd.DataFrame({
    "Column": df.columns,
    "Data_Type": df.dtypes.astype(str).values,
    "Missing_Values": df.isnull().sum().values,
    "Unique_Values": df.nunique().values
})

quality_report

,Column,Data_Type,Missing_Values,Unique_Values
0,OrderID,int64,0,1000
1,OrderDate,datetime64[us],0,510
2,CustomerID,str,0,248
3,Product,str,0,10
4,Category,str,0,4
5,Quantity,int64,0,5
6,UnitPrice,float64,0,870
7,Revenue,float64,0,958
8,Region,str,0,4
9,Status,str,0,3


In [18]:
OUTPUT_FILE = BASE_DIR / "data" / "sales_data_clean.csv"

df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8"
)

print("Clean dataset saved successfully!")
print("File:", OUTPUT_FILE)

Clean dataset saved successfully!
File: d:\پرکاربرد\برنامه\کسب پول و درآمد\تسخیر جزیره\2. Poroject\AI Data Analysis\Separate Project\Power BI\data\sales_data_clean.csv


In [19]:
df_check = pd.read_csv(
    OUTPUT_FILE,
    parse_dates=["OrderDate"]
)

print("Rows:", df_check.shape[0])
print("Columns:", df_check.shape[1])
print("Missing Values:", df_check.isnull().sum().sum())

Rows: 1000
Columns: 13
Missing Values: 0
